# Github
https://github.com/WilliamKA02/CSS_Assignments

# Contributions
All members (William, Tobias & Laura) contributed equally to the assignment.

# Formalia

Please read the [assignment overview page](https://laura.alessandretti.com/comsocsci2026/wiki_pages/Assignments.html) carefully before proceeding. The page contains information about formatting (including formats etc), group sizes, and many other aspects of handing in the assignment. 

__If you fail to follow these simple instructions, it will negatively impact your grade!__

**Due date and time**: The assignment is due on Apr 7th at 23:59. Hand in your Jupyter notebook file (with extension `.ipynb`) via DTU Learn _(Assignment 2)_. 

Remember to include in the first cell of your notebook:
* the link to your group's Git repository 
* group members' contributions

# Imports

In [1]:
import json
import networkx as nx
from networkx.readwrite import json_graph
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

with open("data/Graph.json", "r") as f:
    data = json.load(f)

G = json_graph.node_link_graph(data)

ModuleNotFoundError: No module named 'networkx'

# Part 1: Mixing Patterns and Assortativity

> __Exercise 1: Mixing Patterns and Assortativity__  
>
> __Part 1: Assortativity Coefficient__ 
> 1. *Calculate the Assortativity Coefficient* for the network based on the country of each node. Implement the calculation using the formula provided during the lecture, also available in [this paper](https://arxiv.org/pdf/cond-mat/0209450.pdf) (equation 2, here for directed networks). **Do not use the NetworkX implementation.**

In [ ]:
# First, filter nodes with valid country (non-NaN)
valid_nodes = [node for node in G.nodes() if pd.notna(G.nodes[node].get('country'))]
G_filtered = G.subgraph(valid_nodes).copy()

largest_cc_real = max(nx.connected_components(G_filtered), key=len)

# Create subgraph
G_filtered = G_filtered.subgraph(largest_cc_real).copy()
print(f"LCC nodes: {G_filtered.number_of_nodes()}, edges: {G_filtered.number_of_edges()}")

# Get unique countries
countries = list(set(G_filtered.nodes[node]['country'] for node in G_filtered.nodes()))
country_to_idx = {country: idx for idx, country in enumerate(countries)}
n = len(countries)

# Initialize mixing matrix e_ij (fraction of edges from i to j)
e = np.zeros((n, n))

# Count edges
total_edges = 0
for u, v in G_filtered.edges():
    i = country_to_idx[G_filtered.nodes[u]['country']]
    j = country_to_idx[G_filtered.nodes[v]['country']]
    e[i, j] += 1
    total_edges += 1

# Normalize to fractions
e /= total_edges

# Compute a_i (out-degree fractions) and b_j (in-degree fractions)
term1 = np.trace(e)
a = np.sum(e, axis=1)
b = np.sum(e, axis=0)

# Compute assortativity r
numerator = term1 - np.sum(a**2)
denominator = 1 - np.sum(a**2)
r = numerator / denominator

print(f"sum of ab, a^2 = {np.sum(a*b)}, {np.sum(a*a)} (should be the same)")
print(f"Assortativity coefficient: {r}")

> __Part 2: Configuration model__
> In the following, we are going to assess the significance of the assortativity by comparing the network's assortativity coefficient against that of random networks generated through the configuration model.  
>
> 2. *Implement the configuration model* using the _double edge swap_ algorithm to generate random networks. Ensure each node retains its original degree but with altered connections. Create a function that does that by following these steps:
>
>   - **a.** Create an exact copy of your original network.
>   - **b.** Select two edges, $e_{1} = (u,v)$ and $e_{2} = (x,y)$, ensuring *u != y* and *v != x*.
>   - **c.** Flip the direction of $e_{1}$ to $e_{1} = (v,u)$ 50% of the time. This ensure that your final results is not biased, in case your edges were sorted (they usually are). 
>   - **d.** Ensure that new edges $e_{1}' = (e_{1}[0],e_{2}[1])$ and $e_{2}' = (e_{2}[0],e_{1}[1])$ do not already exist in the network.
>   - **e.** Remove edges $e_{1}$ and $e_{2}$ and add edges $e_{1}'$ and $e_{2}'$.
>   - **f.** Repeat steps **b** to **e** until you have performed $E\cdot10$ swaps, where E is the total number of edges

In [ ]:
import random

def configuration_model(G):
    """
    directed double‑edge‑swap configuration model.
    preserve in‑ and out‑degrees; perform E*10 successful swaps.

    The routine keeps a list of edges and a set for quick membership
    testing.  Additional guards avoid
     * selecting the same edge twice,
     * creating self‑loops after the optional flip,
     * generating two identical new edges (which would introduce
       duplicates into `edges`).
    """
    G_rand = G.copy()
    edges = list(G_rand.edges())
    edge_set = set(edges)          # for O(1) membership checks
    E = len(edges)
    swaps_needed = E * 10
    swaps = 0

    while swaps < swaps_needed and len(edges) >= 2:
        idx1, idx2 = random.sample(range(len(edges)), 2)
        e1 = edges[idx1]
        e2 = edges[idx2]

        u, v = e1
        x, y = e2

        # original self‑loop check
        if u == y or v == x:
            continue

        # flip e1 half the time
        if random.random() < 0.5:
            u, v = v, u

        e1_new = (u, y)
        e2_new = (v, x)

        # skip if either new edge already exists or the two new edges are
        # identical
        if (
            (u,y) in edge_set or (y,u) in edge_set or
            (v,x) in edge_set or (x,v) in edge_set or
            e1_new == e2_new
        ):
            continue

        # perform swap
        G_rand.remove_edge(*e1)
        G_rand.remove_edge(*e2)
        G_rand.add_edge(*e1_new)
        G_rand.add_edge(*e2_new)

        # update bookkeeping
        edge_set.remove(e1); edge_set.remove(e2)
        edge_set.add(e1_new); edge_set.add(e2_new)
        edges[idx1] = e1_new
        edges[idx2] = e2_new

        swaps += 1

    return G_rand

> 3. *Double check that your algorithm works well*, by showing that the degree of nodes in the original network and the new 'randomized' version of the network are the same.

In [ ]:
G_random = configuration_model(G_filtered)

degrees_original = sorted([G_filtered.degree(node) for node in G_filtered.nodes()])
degrees_random = sorted([G_random.degree(node) for node in G_random.nodes()])

print("Degree distribution preserved:", degrees_original == degrees_random)

> __Part 3: Analyzing Assortativity in Random Networks__  
>
> 4. *Generate and analyze at least 100 random networks* using the configuration model. For each, calculate the assortativity with respect to the country and plot the distribution of these values. Compare the results with the assortativity of your original network to determine if connections within the same country are significantly higher than chance.
>

In [ ]:
country_assort_random = []

for i in range(100):
    G_rand = configuration_model(G_filtered)
    r_rand = compute_country_assortativity(G_rand, country_to_idx)
    country_assort_random.append(r_rand)

plt.figure(figsize=(8,5))
plt.hist(country_assort_random, bins=20, density=True)
plt.axvline(r_country, color="red", linestyle="--", label=f"Original r={r_country:.4f}")
plt.xlabel("Country assortativity")
plt.ylabel("Density")
plt.title("Country assortativity in randomized networks")
plt.legend()
plt.show()


> __Part 4: Assortativity by Degree__
>
> 5. *Calculate degree assortativity* for your network using the formula discussed in the lecture.
>

In [ ]:
def compute_degree_assortativity(G):
    edges = list(G.edges())
    M = len(edges)

    deg_u = np.array([G.degree(u) for u, v in edges])
    deg_v = np.array([G.degree(v) for u, v in edges])

    j = np.concatenate([deg_u, deg_v])
    k = np.concatenate([deg_v, deg_u])

    M2 = 2 * M
    mean_jk = np.sum(j * k) / M2
    mean_j = np.sum(j) / M2
    mean_j2 = np.sum(j ** 2) / M2

    r = (mean_jk - mean_j ** 2) / (mean_j2 - mean_j ** 2)
    return r

r_degree_original = compute_degree_assortativity(G_filtered)
print("Degree assortativity:", r_degree_original)

> 6. *Compare your network's degree assortativity* against that of 100 random networks generated via the configuration model. Analyze whether your network shows a tendency for high-degree scientists to connect with other high-degree scientists and vice versa. 

In [ ]:
degree_assort_random = []

for i in range(100):
    G_rand = configuration_model(G_filtered)
    r_rand = compute_degree_assortativity(G_rand)
    degree_assort_random.append(r_rand)

plt.figure(figsize=(8,5))
plt.hist(degree_assort_random, bins=20, density=True)
plt.axvline(r_degree_original, color="red", linestyle="--", label=f"Original r={r_degree_original:.4f}")
plt.xlabel("Degree assortativity")
plt.ylabel("Density")
plt.title("Degree assortativity in randomized networks")
plt.legend()
plt.show()


> __Part 5: Reflection questions__    

> 7. *Assortativity by degree.* Were the results of the degree assortativity in line with your expectations? Why or why not?

The degree Assortativity by degree was negative $r = -0.0883 < 0$, which means it's a slight diassortative network. This is not what we expected, since we thought that popular authors would have a tendencey to write papers with other popular authors. But it seems, that since these authors are so popular, they also get many oppurtunities to work with a wide range of people who are not as popular and who is interested to work with them.

> 8. *Edge flipping.* In the process of implementing the configuration model, you were instructed to flip the edges (e.g., changing $e_1$ from (u,v) to (v,u)) 50% of the time. Why do you think this step is included?

The edge-flipping step is included to remove directional bias introduced by how edges are constructed in the configuration model.

When you build the configuration model, you typically create edges by pairing up "stubs" (half-edges) sequentially from a list. If you always assign the first stub as u and the second as v, then nodes that appear earlier in the stub list will systematically tend to be the source of edges. Even in an undirected graph represented as a directed structure, this creates a subtle ordering artifact.
By flipping each edge with 50% probability, you ensure:

Symmetry — each node is equally likely to appear as u or v in any given edge, so no node is privileged by its position in the stub list.
Uniform sampling — the configuration model is meant to sample uniformly at random from all graphs with a given degree sequence. Without flipping, the sampling distribution is skewed by construction order, meaning you're not truly sampling uniformly.
Independence — flipping breaks correlations between edge direction and node identity that arise purely from the algorithm's sequential pairing process, not from any real graph property.

> 9. *Distribution of assortativity in random networks.* Describe the distribution of degree assortativity values you observed for the random 
networks. Was the distribution pattern expected? Discuss how the nature of random network generation (specifically, the configuration model and edge flipping) might influence this distribution and whether it aligns with theoretical expectations.

The distribution of assortativity coefficients across the random configuration model networks is tightly clustered just below zero, roughly in the range of [-0.02, 0.00], with a clear peak near r ≈ -0.005 to -0.01. This is expected for the following reasons:
Why near zero?
The configuration model randomizes edge connections while preserving the degree sequence. Since high- and low-degree nodes are connected at random, there is no systematic tendency for similar-degree nodes to connect — so the expected assortativity is zero by construction. The tight clustering reflects that, across many random realisations, the mean outcome is effectively neutral mixing.
Why slightly negative rather than exactly zero?
This is a well-known finite-size effect. In networks with a heterogeneous degree distribution (i.e., hubs with very high degree), the configuration model has a slight structural tendency toward disassortativity: high-degree hubs have many connections, making it statistically more likely they connect to low-degree nodes simply because low-degree nodes are more numerous. This bias vanishes as network size → ∞, but is visible in finite graphs.
What the original graph's r = -0.0883 tells us
The original network's assortativity (red dashed line) sits far to the left of the null distribution — well outside the range of any random realisation. This means the disassortativity in your real network is not explained by the degree sequence alone; it reflects a genuine structural property where high-degree nodes preferentially connect to low-degree nodes beyond what randomness would produce.
Role of edge flipping
As discussed, edge flipping ensures uniform sampling from the configuration model ensemble. Without it, a biased sample could artificially shift the distribution, potentially masking or exaggerating the gap between the real and random networks. The tight, symmetric-looking distribution you observe is consistent with proper uniform sampling.

# imports

In [2]:
import ast
from collections import Counter
import numpy as np
import pandas as pd
import networkx as nx
import json

with open("data/Graph_with_communities.json") as f:
    G = nx.node_link_graph(json.load(f))

author_community = {node: data["community"] for node, data in G.nodes(data=True)}
degree_dict = dict(G.degree())

FileNotFoundError: [Errno 2] No such file or directory: 'data/Graph_with_communities.json'

# Part 2: TF-IDF

> __Exercise 1: TF-IDF and the Computational Social Science communities.__ The goal for this exercise is to find the words charachterizing each of the communities of Computational Social Scientists.
> What you need for this exercise: 
>.   
>    * The assignment of each author to their network community, and the degree of each author (Week 6, Exercise 4). This can be stored in a dataframe or in two dictionaries, as you prefer.  
>    * the tokenized _abstract_ dataframe (Week 7, Exercise 2)
>

In [ ]:
df_papers = pd.read_csv("data/CSS_papers.csv")
df_papers["author_ids"] = df_papers["author_ids"].apply(ast.literal_eval)

df_papers_exploded = df_papers.explode("author_ids")
df_papers_exploded["author_id"] = df_papers_exploded["author_ids"].str.replace(
    "https://openalex.org/", "", regex=False
)

df_papers_exploded["community"] = df_papers_exploded["author_id"].map(author_community)
df_papers_exploded = df_papers_exploded.dropna(subset=["community"])
df_papers_exploded["community"] = df_papers_exploded["community"].astype(int)

paper_community = df_papers_exploded.drop_duplicates("id")[["id", "community"]]

df_abstracts = pd.read_csv("data/abstracts_with_tokens.csv")
df_abstracts["tokens"] = df_abstracts["tokens"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
)

df_abstracts["paper_id"] = df_abstracts["id"].str.replace("https://openalex.org/", "", regex=False)
paper_community["paper_id"] = paper_community["id"].str.replace("https://openalex.org/", "", regex=False)

df_merged = df_abstracts.merge(
    paper_community[["paper_id", "community"]],
    on="paper_id",
    how="inner"
)

community_tokens = (
    df_merged[["community", "tokens"]]
    .explode("tokens")
    .dropna()
    .groupby("community")["tokens"]
    .apply(list)
)

> 1. First, check out [the wikipedia page for TF-IDF](https://en.wikipedia.org/wiki/Tf%E2%80%93idf). Explain in your own words the point of TF-IDF. 
>   * What does TF stand for? 

TF-IDF stands for term frequency-inverse document frequency. Its a metric of how important a specific term is in a specific document within a large corpus of documents. The function takes a term $t$ and a document $d$ within the corpus and computes a product of the term-frequency (TF) and the inverse document frequency (IDF).

The TF is simply the frequency of the $t$ in $d$:

$$TF(t,d) = \frac{\text{Count of t in d}}{\text{Total amount of terms in d}}$$


>   * What does IDF stand for?

The IDF is calculated by taking the logarithm of the inverse fraction of the documents containing the term $t$ across the enitre corpus of documents $D$:

$$IDF(t,D) = \log{\frac{\text{Total number of documents}}{\text{Number of documents containing t}}}$$

And finally the TF-IDF is calculated by taking the product of these two metrics. This means the metric measures the importance of a term in a document by calculating how often the term appears in the document and scales it by how much rare the term is across all documents.

> 2. Now, we want to find out which words are important for each *community*, so we're going to create several ***large documents, one for each community***. Each document includes all the tokens of abstracts written by members of a given community. 
>   * Consider a community _c_
>   * Find all the abstracts of papers written by a member of community _c_.
>   * Create a long array that stores all the abstract tokens 
>   * Repeat for all the communities. 
> __Note:__ Here, to ensure your code is efficient, you shall exploit ``pandas`` builtin functions, such as [``groupby.apply``](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.apply.html) or [``explode``](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.explode.html).

In [ ]:
author_community = {node: data["community"] for node, data in G.nodes(data=True)}

# Load papers: each paper has a list of author IDs
df_papers = pd.read_csv("data/CSS_papers.csv")
df_papers["author_ids"] = df_papers["author_ids"].apply(ast.literal_eval)

# Explode so each row is one (paper, author) pair, then strip URL prefix from author ID
df_papers_exploded = df_papers.explode("author_ids")
df_papers_exploded["author_id"] = df_papers_exploded["author_ids"].str.replace("https://openalex.org/", "", regex=False)

# Map author -> community
df_papers_exploded["community"] = df_papers_exploded["author_id"].map(author_community)
df_papers_exploded = df_papers_exploded.dropna(subset=["community"])
df_papers_exploded["community"] = df_papers_exploded["community"].astype(int)

# Keep one community per paper (drop duplicates to avoid counting tokens multiple times)
paper_community = df_papers_exploded.drop_duplicates("id")[["id", "community"]]

# Load abstract dataframe
df_abstracts = pd.read_csv("data/abstracts_with_tokens.csv")

# Parse tokens column (stored as string representation of a list)
df_abstracts["tokens"] = df_abstracts["tokens"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
)

# Merge abstracts with community labels via paper ID (strip URL prefix)
df_abstracts["paper_id"] = df_abstracts["id"].str.replace("https://openalex.org/", "", regex=False)
paper_community["paper_id"] = paper_community["id"].str.replace("https://openalex.org/", "", regex=False)

df_merged = df_abstracts.merge(paper_community[["paper_id", "community"]], on="paper_id", how="inner")

# Build one large token array per community using explode + groupby
community_tokens = (
    df_merged[["community", "tokens"]]
    .explode("tokens")
    .dropna(subset=["tokens"])
    .groupby("community")["tokens"]
    .apply(list)
)

print(f"Number of communities: {len(community_tokens)}")
print(f"\nToken counts per community (top 10):")
print(community_tokens.apply(len).sort_values(ascending=False).head(10))
print(f"\nSample tokens from community 0: {community_tokens[0][:10]}")

> 3. Now, we're ready to calculate the TF for each word. Use the method of your choice to find the top 5 terms within the __top 5 communities__ (by number of authors). 

In [ ]:

# --- Top 5 communities by number of authors ---
author_df = pd.DataFrame(
    [(node, data["community"]) for node, data in G.nodes(data=True)],
    columns=["author_id", "community"]
)
top5_communities = (
    author_df.groupby("community")["author_id"]
    .count()
    .sort_values(ascending=False)
    .head(5)
    .index.tolist()
)
print("Top 5 communities by author count:", top5_communities)

# --- TF: term frequency per community ---
# TF(t, c) = count(t in c) / total tokens in c
def compute_tf(tokens):
    counts = Counter(tokens)
    total = len(tokens)
    return {word: count / total for word, count in counts.items()}

community_tf = {c: compute_tf(community_tokens[c]) for c in community_tokens.index}

print("\nTop 5 TF terms per community (top 5 communities by author count):")
for c in top5_communities:
    top5 = sorted(community_tf[c].items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"  Community {c}: {[w for w, _ in top5]}")


>   * Describe similarities and differences between the communities.

You can clearly observe that the communities tend to share terms with high TF like "use", "learn", "model" and "effect" since these are very generic terms that will appear in many settings and papers. It does however seem like the two biggest communities revolve around social experiments and studies given the terms "commun", "experi", "group" and "people", where the three other communities tend to focus more on machine learning / deep learning with terms as "imag", "method", "model" and "learn".

>   * Why aren't the TFs not necessarily a good description of the communities?

TF only measures how frequently a word appears within a community, and it does not account for whether that word is equally common across all communities. Like mentioned above, the term "use" is in almost all of the communites, and it does tell anything specific about the community, since its so generic. When computing the IDF, it was observed that "use" had the lowest score, meaning it was observed in most of the communities.

>   * Next, we calculate IDF for every word. 

In [ ]:
N = len(community_tokens)
all_words = set(w for tokens in community_tokens for w in tokens)

doc_freq = Counter()
for tokens in community_tokens:
    for word in set(tokens):
        doc_freq[word] += 1

idf = {word: np.log(N / doc_freq[word]) for word in all_words}

print(f"IDF computed for {len(idf)} words across {N} communities.")

>   * What base logarithm did you use? Is that important?

We used the natural logarithm (`np.log`), which computes $\log_e$. The choice of base does not affect the ranking of words — it only scales all IDF values by a constant factor ($\log_e x = \log_{10} x \cdot \ln 10$). Since TF-IDF is used for ranking, the log base is irrelevant in practice.

> 4. We're ready to calculate TF-IDF. Do that for the __top 9 communities__ (by number of authors). Then for each community: 
>   * List the 10 top TF words 
>   * List the 10 top TF-IDF words
>   * List the top 3 authors (by degree)

In [ ]:
# --- Top 9 communities by number of authors ---
top9_communities = (
    author_df.groupby("community")["author_id"]
    .count()
    .sort_values(ascending=False)
    .head(9)
    .index.tolist()
)

# --- TF-IDF = TF * IDF ---
community_tfidf = {
    c: {word: tf * idf[word] for word, tf in community_tf[c].items()}
    for c in community_tokens.index
}

# --- Top 3 authors by degree per community ---
degree_dict = dict(G.degree())

for c in top9_communities:
    # Top 10 TF words
    top10_tf = sorted(community_tf[c].items(), key=lambda x: x[1], reverse=True)[:10]
    # Top 10 TF-IDF words
    top10_tfidf = sorted(community_tfidf[c].items(), key=lambda x: x[1], reverse=True)[:10]
    # Top 3 authors by degree in this community
    community_authors = author_df[author_df["community"] == c]["author_id"].tolist()
    top3_authors = sorted(community_authors, key=lambda a: degree_dict.get(a, 0), reverse=True)[:3]
    top3_names = [G.nodes[a]["display_name"] for a in top3_authors]

    print(f"\n{'='*60}")
    print(f"Community {c}  (top 3 authors by degree: {top3_names})")
    print(f"  Top 10 TF      : {[w for w, _ in top10_tf]}")
    print(f"  Top 10 TF-IDF  : {[w for w, _ in top10_tfidf]}")


>   * Are these 10 words more descriptive of the community? If yes, what is it about IDF that makes the words more informative?

Yes, as mentioned above the top TF words are very similar across communities. Generic terms like "use" and "model" dominate the TF list — because they appear frequently in every community's abstracts. They describe CSS as a whole, not any specific subfield.

The TF-IDF words are far more specific: communities focusing on social / psychological science contains terms like 'emot', 'peopl', 'belief', 'psycholog' instead of the TF terms: 'effect', 'group', 'studi' which are much less specific. These words match the known expertise of the top authors by degree, providing a validation that the method is working.

IDF down-weights words that appear in many communities (high document frequency → low IDF → near-zero TF-IDF). This cancels out the shared vocabulary that dominates TF alone. Only words that are frequent within a community and rare across communities receive a high TF-IDF score — exactly the words that are distinctive to that community.


 __Exercise 2: The Wordcloud__. It's time to visualize our results!

> * Install the [`WordCloud`](https://pypi.org/project/wordcloud/) module. 
> * Now, create word-cloud for each community. Feel free to make it as fancy or non-fancy as you like.
> * Make sure that, together with the word cloud, you print the names of the top three authors in each community (see my plot above for inspiration). 

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(18, 18))

for ax, c in zip(axes.flatten(), top9_communities):
    # Build word cloud from TF-IDF scores
    wc = WordCloud(
        width=500, height=380,
        background_color="white",
        colormap="RdPu",
        max_words=60,
        prefer_horizontal=0.7,
    ).generate_from_frequencies(community_tfidf[c])

    # Top 3 authors by degree in this community
    community_authors = author_df[author_df["community"] == c]["author_id"].tolist()
    top3_authors = sorted(
        community_authors, key=lambda a: degree_dict.get(a, 0), reverse=True
    )[:3]
    top3_names = [G.nodes[a]["display_name"] for a in top3_authors]

    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title("\n".join(top3_names), fontsize=9, va="bottom", pad=8)

plt.suptitle("TF-IDF Word Clouds per Community", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

> * Comment on your results. What can you conclude on the different sub-communities in Computational Social Science? 

The nine communities map onto clearly distinct sub-fields of Computational Social Science:

| Community (top author) | Dominant words | Sub-field |
|---|---|---|
| Bernstein / De Choudhury / Kleinberg | *user, task, interact, social_media, design* | Human-Computer Interaction & social computing |
| Ariely / Haslam / Markus | *emoti, peopl, belief, stereotyp, social_ident* | Social psychology & behavioral science |
| Jiebo Luo / Chunhua Shen / Bolei Zhou | *video, imag, learn, cnn, dataset* | Computer vision & deep learning |
| Jurafsky / Lee / Callison-Burch | *languag, sentence, text, task, machin_translat* | Natural language processing |
| Lewandowsky / Rand / Hertwig | *memori, cognit, choic, children, judgment* | Cognitive psychology & decision-making |
| Amaral / Stanley / Kertész | *power_law, network, market, percol, fluctuat* | Statistical physics & complex systems |
| Menczer / Ahn / Flammini | *citat, twitter, social_network, vaccin, epidem* | Online misinformation & science of science |
| Quercia / Wilson / Gummadi | *attack, privaci, secur, fair, app* | Cybersecurity, fairness & privacy |
| Pentland / Lepri / Sebe | *facial_express, recognit, video, robot* | Affective computing & human sensing |

The results suggest that Computational Social Science is made up of several distinct but related sub-communities. Some communities are centered around natural language processing, some around computer vision, some around social psychology, and others around network science or misinformation. The word clouds and TF-IDF terms make these differences visible.

Overall, the analysis shows that Computational Social Science is highly interdisciplinary. The methods and topics differ across communities, but they are all connected through shared computational approaches and collaboration patterns.


> * Look up online the top author in each community. In light of your search, do your results make sense?



- **Bernstein & Kleinberg** are respectively known for crowdsourcing/HCI (Stanford) and foundational network+algorithm work (Cornell) — the *user/task/interact* words are a perfect fit.
- **Dan Ariely** is famous for behavioral economics (*irrational* decision-making), Haslam and Markus for social identity theory — *stereotyp*, *social_ident*, and *belief* are exactly their vocabulary.
- **Jurafsky** co-authored the standard NLP textbook; *machin_translat*, *languag*, and *sentence* are his core topics.
- **H. Eugene Stanley and Amaral** are pioneers of econophysics and complex systems — *power_law*, *percolation*, and *financi_market* are hallmarks of their work.
- **Lewandowsky** studies memory and misinformation; *memori*, *cognit*, and *children* align with his cognitive-psychology focus.
- **Menczer** runs the Observatory on Social Media (OSoMe) at Indiana — *twitter*, *vaccin*, *epidem* directly reflect his misinformation-spread research.
- **Gummadi and Wilson** are known for fairness and privacy in online platforms — *attack*, *secur*, *fair* match precisely.
- **Pentland** leads MIT's Human Dynamics group; *facial_express* and *recognit* fit his work on behavioral sensing.

Overall, TF-IDF successfully recovered coherent, interpretable research communities purely from abstract text — and the top authors in each community are exactly the scholars you would expect to find there.

 __Exercise 3: Computational Social Science__ 

> * In light of your data-driven analysis, has your understanding of the field changed? How? 

Our understanding of Computational Social Science has changed through this analysis. Before, we thought of the field as relatively unified, but the network and text analysis show that it consists of several quite different sub-communities. Some are much closer to computer science, while others are rooted in psychology, sociology, or physics. At the same time, the collaboration network shows that these communities still belong to the same broader field. This makes Computational Social Science seem less like one narrow discipline and more like an interdisciplinary meeting point.